# 06 — SQL Joins and Aggregations

### Why this exists

Joins combine row sets, but the join changes row counts. Many data bugs are really grain bugs that only become visible after aggregation.

### Learning objectives

- state the grain before joining
- detect row multiplication
- aggregate at the intended grain

### Prerequisite

05 — SQL Fundamentals

### Engineering loop

**Why → Sketch → Predict → Build → Measure → Break → Diagnose → Improve → Production**


## Capability contract

**Capability:** C02 — Data Access Querying & Modeling  
**Workstream:** Revenue Data Platform  
**Primary roles:** Data Engineer|Analytics Engineer|Platform Engineer  
**You will prove:** Translate a business question into correct data access and defend the model/query behavior.


## 🖊️ Sketch note

![Sketch note — SQL Joins & Aggregations](../assets/sketch-notes/06-sql-joins-aggregations.svg)

> **Visual-first rule:** spend 30–60 seconds explaining the sketch in your own words before reading the implementation. Then return to it after the experiment and redraw it from memory.


## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S01 — Ticket DATA-1042: Revenue dashboard mismatch** → [`../work_simulations/01_acme_commerce_revenue_mismatch/README.md`](../work_simulations/01_acme_commerce_revenue_mismatch/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Predict before you run

Write down one thing you expect to observe and why. Do not use the output as your prediction.


## Production bridge

Ask: what happens when the data is 100× larger, arrives late, changes shape, or is queried by several teams? State the assumption that makes your experiment useful.


## Independent challenge

Construct a deliberately many-to-many join and prove why the aggregate changes.

### Evidence to keep

- your prediction
- one measurement
- the failure or edge case you tested
- one production implication


# 06 — SQL Joins and Aggregations

## The minimum theory

Real systems keep related facts in separate tables. A **join** combines them through a relationship.

Example:

```text
customers                 orders
customer_id  ───────────> customer_id
one customer               many orders
```

The critical idea is **grain**: what does one row represent?

A one-to-many join can multiply rows. If you aggregate without understanding that multiplication, metrics can be wrong.

`INNER JOIN` keeps matching rows. `LEFT JOIN` keeps every row from the left table.


In [ ]:
import duckdb,pandas as pd
customers=pd.DataFrame({"customer_id":[1,2,3,4],"name":["Asha","Ben","Chen","Diya"],"country":["IN","US","DE","IN"]})
orders=pd.DataFrame({"order_id":[101,102,103,104,105,106],"customer_id":[1,1,2,3,3,3],"amount":[100,150,80,200,50,120]})
con=duckdb.connect(); con.register("customers",customers); con.register("orders",orders)


In [ ]:
print(con.sql("SELECT c.customer_id,c.name,o.order_id,o.amount FROM customers c JOIN orders o ON c.customer_id=o.customer_id ORDER BY c.customer_id,o.order_id").df())
print(con.sql("SELECT c.customer_id,c.name,o.order_id,o.amount FROM customers c LEFT JOIN orders o ON c.customer_id=o.customer_id ORDER BY c.customer_id").df())


In [ ]:
con.sql("SELECT c.customer_id,c.name,COALESCE(SUM(o.amount),0) AS revenue FROM customers c LEFT JOIN orders o ON c.customer_id=o.customer_id GROUP BY c.customer_id,c.name ORDER BY revenue DESC").df()


## Break it: many-to-many

Create `order_tags` with multiple tags for one order. Join it to orders and calculate `SUM(amount)`.

If an order has multiple tags, its amount can appear multiple times.

**Engineering challenge:** demonstrate the inflated total and fix it.

This is why production SQL starts by stating and checking table grain.

**Next:** window functions.
